# Day 15 — Assignment Problem & Linear Programming Basics

**Goal:** Build intuition for why a solver beats greedy assignment,
before writing any OR-Tools code (that starts Day 16).

**Context:** `score_matrix.csv` gives us a score for every
(employee, project, role) triple. Picking the top scorer for each
role independently (greedy) can double-book the same strong
candidate across multiple roles, leaving worse people to fill the
gaps — even though a different combination would score higher overall.

## Example 1 — Greedy happens to work

|       | Role A | Role B |
|-------|--------|--------|
| Alice | 0.9    | 0.4    |
| Bob   | 0.6    | 0.8    |
| Carol | 0.5    | 0.5    |

**Greedy:** Role A → Alice (0.9), Role B → Bob (0.8) → total **1.7**

Check all valid combinations (no person used twice) below.

In [1]:
from itertools import permutations

people = ["Alice", "Bob", "Carol"]
roles  = ["RoleA", "RoleB"]

scores = {
    ("Alice", "RoleA"): 0.9, ("Alice", "RoleB"): 0.4,
    ("Bob",   "RoleA"): 0.6, ("Bob",   "RoleB"): 0.8,
    ("Carol", "RoleA"): 0.5, ("Carol", "RoleB"): 0.5,
}

best_total = -1
best_combo = None

for combo in permutations(people, len(roles)):
    total = sum(scores[(p, r)] for p, r in zip(combo, roles))
    print(f"{roles[0]}→{combo[0]:<6} {roles[1]}→{combo[1]:<6}  total={total:.2f}")
    if total > best_total:
        best_total = total
        best_combo = combo

print(f"\n Best assignment: {roles[0]}→{best_combo[0]}, {roles[1]}→{best_combo[1]}  (total={best_total:.2f})")

RoleA→Alice  RoleB→Bob     total=1.70
RoleA→Alice  RoleB→Carol   total=1.40
RoleA→Bob    RoleB→Alice   total=1.00
RoleA→Bob    RoleB→Carol   total=1.10
RoleA→Carol  RoleB→Alice   total=0.90
RoleA→Carol  RoleB→Bob     total=1.30

 Best assignment: RoleA→Alice, RoleB→Bob  (total=1.70)


## Example 2 — Greedy fails

|       | Role A | Role B |
|-------|--------|--------|
| Alice | 0.9    | 0.85   |
| Bob   | 0.3    | 0.2    |

**Greedy:** Role A's top scorer is Alice (0.9) → assign her.
Role B's top scorer is *also* Alice (0.85) — but she's taken, so
Role B falls to Bob (0.2).

Greedy total = 0.9 + 0.2 = **1.1**

But Alice→B, Bob→A gives 0.85 + 0.3 = **1.15** — higher!

This is exactly our real situation: in `score_matrix.csv`, E075 and
E066 are top-2 candidates for "Backend Dev" across *multiple*
projects (P001, P002, P003). Greedily assigning per-role
independently risks leaving a better global combination on the table.

In [2]:
people = ["Alice", "Bob"]
roles  = ["RoleA", "RoleB"]

scores = {
    ("Alice", "RoleA"): 0.9, ("Alice", "RoleB"): 0.85,
    ("Bob",   "RoleA"): 0.3, ("Bob",   "RoleB"): 0.2,
}

# Greedy: assign each role to its independent top scorer
greedy_assignment = {}
for r in roles:
    top_person = max(people, key=lambda p: scores[(p, r)])
    greedy_assignment[r] = top_person

greedy_total = sum(scores[(greedy_assignment[r], r)] for r in roles)
print("Greedy (naive) assignment:")
for r, p in greedy_assignment.items():
    print(f"  {r} → {p} ({scores[(p, r)]})")
print(f"  Total: {greedy_total:.2f}\n")

# Brute force: check all valid combinations
best_total = -1
best_combo = None
for combo in permutations(people, len(roles)):
    total = sum(scores[(p, r)] for p, r in zip(combo, roles))
    if total > best_total:
        best_total = total
        best_combo = combo

print("Optimal assignment:")
for r, p in zip(roles, best_combo):
    print(f"  {r} → {p} ({scores[(p, r)]})")
print(f"  Total: {best_total:.2f}")

print(f"\n{'  Greedy is suboptimal!' if greedy_total < best_total else ' Greedy matched optimal here.'}")

Greedy (naive) assignment:
  RoleA → Alice (0.9)
  RoleB → Alice (0.85)
  Total: 1.75

Optimal assignment:
  RoleA → Bob (0.3)
  RoleB → Alice (0.85)
  Total: 1.15

 Greedy matched optimal here.


## Example 3 

A 3-person × 3-role score table where greedy fails.
Make one person the top (or near-top) scorer for at least
two roles, with a close second-best elsewhere.


In [ ]:
#3x3 example
people = ["Person1", "Person2", "Person3"]
roles  = ["RoleA", "RoleB", "RoleC"]

scores = {
    ("Person1", "RoleA"): 0.0, ("Person1", "RoleB"): 0.0, ("Person1", "RoleC"): 0.0,
    ("Person2", "RoleA"): 0.0, ("Person2", "RoleB"): 0.0, ("Person2", "RoleC"): 0.0,
    ("Person3", "RoleA"): 0.0, ("Person3", "RoleB"): 0.0, ("Person3", "RoleC"): 0.0,
}

# Greedy
greedy_assignment = {}
for r in roles:
    top_person = max(people, key=lambda p: scores[(p, r)])
    greedy_assignment[r] = top_person
greedy_total = sum(scores[(greedy_assignment[r], r)] for r in roles)

# Brute force optimal
best_total = -1
best_combo = None
for combo in permutations(people, len(roles)):
    total = sum(scores[(p, r)] for p, r in zip(combo, roles))
    if total > best_total:
        best_total = total
        best_combo = combo

print(f"Greedy total:  {greedy_total:.2f}  →", greedy_assignment)
print(f"Optimal total: {best_total:.2f}  →", dict(zip(roles, best_combo)))
print(f"\n{'  Greedy is suboptimal!' if greedy_total < best_total else 'Greedy matched optimal — try different numbers.'}")

Greedy total:  0.00  → {'RoleA': 'Person1', 'RoleB': 'Person1', 'RoleC': 'Person1'}
Optimal total: 0.00  → {'RoleA': 'Person1', 'RoleB': 'Person2', 'RoleC': 'Person3'}

Greedy matched optimal — try different numbers.


##  Day 15 Takeaways

- Greedy (top scorer per role, independently) can leave value on
  the table when one person is a strong candidate for multiple roles.
- The brute-force check above (`itertools.permutations`) only works
  for tiny examples — with 80 employees × dozens of role-slots, this
  is computationally impossible (factorial blowup).
- This is exactly why we need a real solver: **Day 16** installs
  OR-Tools and replaces this brute-force loop with CP-SAT, which
  solves the same kind of problem at scale, efficiently.

# Day 16 — OR-Tools Install + Toy CP-SAT Script

**Goal:** Install Google OR-Tools and solve the Day 15 toy examples
using the real CP-SAT solver, to confirm it matches our brute-force
answers — then trust it to scale where brute force can't.

In [5]:
from ortools.sat.python import cp_model

## Solve Day 15's Example 2 with CP-SAT

Recall: Alice/Bob across RoleA/RoleB, where greedy was suboptimal
(greedy=1.1, optimal=1.15). Let's confirm CP-SAT finds 1.15.

In [6]:
from ortools.sat.python import cp_model

# CP-SAT needs integers, so scale scores by 100
scores = {
    ("Alice", "RoleA"): 90, ("Alice", "RoleB"): 85,
    ("Bob",   "RoleA"): 30, ("Bob",   "RoleB"): 20,
}
people = ["Alice", "Bob"]
roles  = ["RoleA", "RoleB"]

model = cp_model.CpModel()

# Decision variables: x[person, role] = 1 if assigned
x = {}
for p in people:
    for r in roles:
        x[p, r] = model.NewBoolVar(f"x_{p}_{r}")

# Constraint: each role filled by exactly 1 person
for r in roles:
    model.Add(sum(x[p, r] for p in people) == 1)

# Constraint: each person assigned to at most 1 role
for p in people:
    model.Add(sum(x[p, r] for r in roles) <= 1)

# Objective: maximize total score
model.Maximize(sum(scores[p, r] * x[p, r] for p in people for r in roles))

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(f"Status: {solver.StatusName(status)}  |  Total score: {solver.ObjectiveValue() / 100:.2f}")
    for p in people:
        for r in roles:
            if solver.Value(x[p, r]):
                print(f"  {p} → {r} (score {scores[p, r] / 100:.2f})")
else:
    print("No solution found")

Status: OPTIMAL  |  Total score: 1.15
  Alice → RoleB (score 0.85)
  Bob → RoleA (score 0.30)


##  Day 16 Takeaways

- CP-SAT total should print **1.15** with Alice→RoleB, Bob→RoleA —
  matching our Day 15 brute-force answer exactly.
- This is the same model structure we'll scale up to all 80
  employees × all project roles on Day 18–19. The only thing that
  changes is the size of `people`, `roles`, and `scores` — the
  variables/constraints/objective pattern stays identical.

# Day 17 — Problem Formulation for the Real Staffing Optimizer

**Decision variables:**
`x[employee_id, project_id, role]` = 1 if that employee is assigned
to that role on that project, else 0.

**Objective:**
Maximize `sum(final_score[e,p,r] * x[e,p,r])` over all eligible
(e, p, r) triples from `score_matrix.csv`.

**Constraints:**
1. **One person per role-slot** (headcount=1 assumed for now):
   `sum(x[e,p,r] for e in employees) == 1` for each (p, r).
2. **No double-booking**: each employee assigned to **at most one**
   role total across all projects in this run.
   `sum(x[e,p,r] for all p,r) <= 1` for each employee e.
3. **Eligibility**: only create `x[e,p,r]` variables for rows where
   `eligible == True` in `score_matrix.csv` — ineligible pairs never
   become variables at all.
4. **Unstaffable roles**: if a (project, role) slot has zero eligible
   candidates, flag it *before* solving — the solver can't fix a
   data problem.

This formulation is what `optimize_staffing.py` implements directly
on Day 18–19. Keep this cell — you'll reference it again on Day 21.